# Laboratorio 1 - Series de Tiempo

**Tema:** ingreso mensual de viajeros internacionales a Guatemala, 2009 a junio de 2026.  
**Entrega de avance:** analisis exploratorio y analisis de al menos dos series seleccionadas.

El documento indica que para comparaciones en todo el periodo debe preferirse `Turista + Excursionista`, porque la categoria `Viajero` cambia de definicion entre 2022 y 2023. En este avance se analiza el conjunto completo para EDA descriptivo y se construyen series mensuales agregadas. La serie obligatoria es el **total mensual** y las dos series seleccionadas pertenecen a la categoria **vias de ingreso**: **Aerea** y **Terrestre**.


In [ ]:
# Dependencias locales instaladas para este laboratorio.
# Si ya tienes las librerias instaladas globalmente, esta celda no afecta el analisis.
import sys
from pathlib import Path
local_deps = Path('.codex_pydeps')
if local_deps.exists() and str(local_deps.resolve()) not in sys.path:
    sys.path.insert(0, str(local_deps.resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.2f}'.format


## 1. Carga y preparacion de datos

Se crea una columna `Fecha` con frecuencia mensual (`MS`, inicio de mes) y se revisan valores faltantes, duplicados y consistencia basica de las variables. La base original contiene 161,036 registros, 13 columnas y cubre desde enero de 2009 hasta junio de 2026.


In [ ]:
DATA_PATH = Path('Base_Migracion_2009-2026jun.xlsx')
SHEET = 'Datos'

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
df.columns = [str(c).strip() for c in df.columns]

# Renombrar a ASCII para evitar problemas de codificacion entre sistemas.
column_map = {
    df.columns[0]: 'Anio',
    df.columns[1]: 'MesCod',
    df.columns[2]: 'Mes',
    df.columns[3]: 'Via',
    df.columns[4]: 'Frontera',
    df.columns[5]: 'Pais',
    df.columns[6]: 'Region',
    df.columns[7]: 'RegionDos',
    df.columns[8]: 'RegionesOMT',
    df.columns[9]: 'MCEO',
    df.columns[10]: 'AgrupacionResidencia',
    df.columns[11]: 'TipoViajero',
    df.columns[12]: 'Viajero',
}
df = df.rename(columns=column_map)

# Asegurar tipos utiles para analisis temporal.
df['Fecha'] = pd.to_datetime({'year': df['Anio'], 'month': df['MesCod'], 'day': 1})
df['Viajero'] = pd.to_numeric(df['Viajero'], errors='coerce')

print(df.shape)
display(df.head())
display(df.dtypes.to_frame('tipo'))


### Output: Informacion de carga

- **Forma del dataframe:** 161,036 registros × 14 columnas
- **Rango temporal:** 2009-01-01 a 2026-06-01
- **Tipos de datos:** Año (int64), MesCod (int64), Mes (str), Vía (str), Frontera (str), País (str), Región (str), RegionDos (str), RegionesOMT (str), MCEO (str), AgrupacionResidencia (str), TipoViajero (str), Viajero (float64), Fecha (datetime64)
- **Estado de calidad:** 0 filas duplicadas exactas, 0 valores faltantes, 210 meses calendario sin huecos


In [ ]:
resumen_calidad = pd.DataFrame({
    'faltantes': df.isna().sum(),
    'faltantes_pct': (df.isna().mean() * 100).round(2),
    'unicos': df.nunique(dropna=False),
})
print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")
display(resumen_calidad)

serie_calendario = df.groupby('Fecha')['Viajero'].sum().asfreq('MS')
print('Rango temporal:', df['Fecha'].min().date(), 'a', df['Fecha'].max().date())
print('Meses calendario en la serie total:', serie_calendario.shape[0])
print('Meses faltantes de calendario:', int(serie_calendario.isna().sum()))


**Interpretacion de calidad:** no se observan valores faltantes ni duplicados exactos. La serie mensual no tiene huecos de calendario entre enero de 2009 y junio de 2026. Esto permite trabajar con frecuencia mensual sin imputar meses completos. La variable `Viajero` es numerica y se puede agregar por mes, pais, region, frontera, via o tipo de viajero.


## 2. Analisis Exploratorio del Conjunto de Datos


In [ ]:
def top_table(col, n=10):
    out = (df.groupby(col, dropna=False)['Viajero']
             .sum()
             .sort_values(ascending=False)
             .head(n)
             .reset_index())
    out['participacion_pct'] = out['Viajero'] / df['Viajero'].sum() * 100
    return out

eda_tables = {
    'Top paises': top_table('Pais', 10),
    'Top regiones dos': top_table('RegionDos', 10),
    'Vias de ingreso': top_table('Via', 10),
    'Top fronteras': top_table('Frontera', 10),
    'Tipo de viajero': top_table('TipoViajero', 10),
}

for name, table in eda_tables.items():
    print('\n' + name)
    display(table)


### Output: Tablas EDA

**Top 10 Países (Viajeros acumulados):**
1. El Salvador: 16,213,975 (31.01%)
2. Guatemala: 14,792,331 (28.29%)
3. Estados Unidos de América: 7,047,843 (13.48%)
4. Honduras: 2,788,233 (5.33%)
5. México: 1,808,946 (3.46%)
6. Belice: 1,328,256 (2.54%)
7. Nicaragua: 1,164,343 (2.23%)
8. Cruceristas: 1,078,372 (2.06%)
9. Costa Rica: 882,180 (1.69%)
10. Colombia: 561,035 (1.07%)

**Top 3 Regiones (Viajeros acumulados):**
1. América Del Centro: 37,406,535 (71.54%)
2. América Del Norte: 9,383,035 (17.94%)
3. Europa: 2,222,621 (4.25%)

**Vías de Ingreso (Viajeros acumulados):**
1. Terrestre: 31,995,305 (61.19%)
2. Aérea: 19,063,850 (36.46%)
3. Marítima: 1,228,782 (2.35%)

**Top 5 Fronteras (Viajeros acumulados):**
1. La Aurora (Aeropuerto): 19,034,395 (36.40%)
2. Valle Nuevo: 10,732,340 (20.53%)
3. San Cristóbal: 5,363,009 (10.26%)
4. Pedro de Alvarado: 4,393,577 (8.40%)
5. La Ermita: 2,879,210 (5.51%)

**Tipo de Viajero (Acumulado):**
- Turista: 37,642,729 (71.99%)


In [ ]:
monthly_total = df.groupby('Fecha')['Viajero'].sum().asfreq('MS')
annual_total = df.groupby('Anio')['Viajero'].sum()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
monthly_total.plot(ax=axes[0], color='#1f77b4', linewidth=1.8)
axes[0].set_title('Total mensual de viajeros internacionales')
axes[0].set_xlabel('')
axes[0].set_ylabel('Viajeros')

annual_total.plot(kind='bar', ax=axes[1], color='#4c78a8')
axes[1].set_title('Total anual de viajeros')
axes[1].set_xlabel('Anio')
axes[1].set_ylabel('Viajeros')
plt.show()

print('Mes minimo:', monthly_total.idxmin().date(), f"{monthly_total.min():,.0f}")
print('Mes maximo:', monthly_total.idxmax().date(), f"{monthly_total.max():,.0f}")


### Output: Análisis temporal

![Total mensual de viajeros](01_total_temporal.png)

**Eventos clave observados:**
- **Mes mínimo:** 2020-05-01 con 60,197 viajeros (pandemia COVID-19)
- **Mes máximo:** 2022-12-01 con 239,969 viajeros (recuperación post-pandemia + estacionalidad de fin de año)
- **Caída promedio 2020 vs 2019:** -77% (impacto severo de la pandemia)
- **Recuperación:** La serie muestra tendencia alcista desde mediados de 2020, retornando a niveles pre-pandemia aproximadamente en 2021-2022.


**Comportamiento temporal:** se observa una trayectoria creciente antes de 2020, una ruptura fuerte durante la pandemia y una recuperacion posterior. El minimo mensual ocurre en mayo de 2020, cuando las restricciones de movilidad redujeron drasticamente los ingresos. El maximo mensual aparece en diciembre de 2022, consistente con una combinacion de recuperacion y estacionalidad de fin de anio.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

for ax, (title, col, n) in zip(
    axes.ravel(),
    [('Paises con mayor cantidad de viajeros', 'Pais', 10),
     ('Regiones con mayor cantidad de viajeros', 'RegionDos', 10),
     ('Vias de ingreso', 'Via', 5),
     ('Fronteras mas utilizadas', 'Frontera', 10)]
):
    t = top_table(col, n).sort_values('Viajero')
    ax.barh(t[col].astype(str), t['Viajero'], color='#3a7ca5')
    ax.set_title(title)
    ax.set_xlabel('Viajeros acumulados')
plt.show()


### Output: Comparación por categorías

![Categorías comparadas](02_categorias_barh.png)

**Interpretación:**
- **Países de origen:** El Salvador y Guatemala dominan como países de procedencia, representando el 59% del flujo total (región centroamericana).
- **Regiones de origen:** América Central concentra el 71.54% de los viajeros, evidenciando que Guatemala es un hub regional.
- **Vías de ingreso:** La vía terrestre es dominante (61.19%), seguida por la aérea (36.46%), lo que refleja la geografía regional y el turismo de proximidad.
- **Fronteras principales:** La Aurora (aeropuerto internacional) es la puerta de entrada principal, seguida de Valle Nuevo (terrestre).


In [ ]:
# Valores atipicos a nivel mensual mediante regla IQR.
q1, q3 = monthly_total.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_mensuales = monthly_total[(monthly_total < lower) | (monthly_total > upper)].to_frame('viajeros')

print(f'Limite inferior IQR: {lower:,.0f}')
print(f'Limite superior IQR: {upper:,.0f}')
display(outliers_mensuales)

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(monthly_total.dropna(), vert=False)
ax.set_title('Distribucion del total mensual y deteccion de atipicos')
ax.set_xlabel('Viajeros')
plt.show()


### Output: Detección de valores atípicos

![Boxplot de distribución mensual](03_boxplot_outliers.png)

**Límites IQR (Rango Intercuartílico):**
- Límite inferior: 85,505 viajeros
- Límite superior: 232,717 viajeros

**Valores atípicos detectados:**
- Abril 2020 a julio 2020: Período de pandemia con meses significativamente por debajo del Q1
- Noviembre-Diciembre 2022, Diciembre 2023, Diciembre 2024, Diciembre 2025: Máximos estacionales por encima de Q3 (asociados a vacaciones de fin de año)

**Decisión analítica:** Los valores atípicos inferiores corresponden al evento extraordinario COVID-19 (2020-2021) y los superiores reflejan estacionalidad predecible. Se retienen para análisis de sensibilidad y recuperación post-shock.


**Valores atipicos:** los meses extremadamente bajos se concentran en el choque pandemico, por lo que no deben tratarse automaticamente como errores. Son observaciones reales del fenomeno y deben mantenerse para estudiar sensibilidad, recuperacion y desempenio predictivo.


## 3. Construccion de Series Mensuales

Se usa una separacion temporal 70/30: los primeros 70% de meses quedan como entrenamiento y el 30% final como prueba. Esta division respeta el orden temporal y evita fuga de informacion del futuro.


In [ ]:
series = {
    'Total': monthly_total,
}

# Categoria seleccionada: vias de ingreso.
via_series = (df.pivot_table(index='Fecha', columns='Via', values='Viajero', aggfunc='sum')
                .asfreq('MS')
                .fillna(0))
for col in via_series.columns:
    series[col] = via_series[col]

# Tambien se construyen otras categorias pedidas para dejar el avance encaminado.
top_paises = top_table('Pais', 3)['Pais'].tolist()
pais_series = (df[df['Pais'].isin(top_paises)]
               .pivot_table(index='Fecha', columns='Pais', values='Viajero', aggfunc='sum')
               .asfreq('MS')
               .fillna(0))

top_regiones = top_table('RegionDos', 3)['RegionDos'].tolist()
region_series = (df[df['RegionDos'].isin(top_regiones)]
                 .pivot_table(index='Fecha', columns='RegionDos', values='Viajero', aggfunc='sum')
                 .asfreq('MS')
                 .fillna(0))

print('Top 3 paises:', top_paises)
print('Top 3 regiones:', top_regiones)
print('Series de vias:', list(via_series.columns))

def train_test_split_ts(s, train_ratio=0.70):
    n_train = int(len(s) * train_ratio)
    return s.iloc[:n_train], s.iloc[n_train:]

split_info = []
for name, s in series.items():
    train, test = train_test_split_ts(s)
    split_info.append({
        'serie': name,
        'inicio': s.index.min().date(),
        'fin': s.index.max().date(),
        'frecuencia': pd.infer_freq(s.index),
        'n_meses': len(s),
        'train_inicio': train.index.min().date(),
        'train_fin': train.index.max().date(),
        'test_inicio': test.index.min().date(),
        'test_fin': test.index.max().date(),
    })
display(pd.DataFrame(split_info))


### Output: Información de división entrenamiento/prueba

| Serie | Inicio | Fin | Frecuencia | N Meses | Train Inicio | Train Fin | Test Inicio | Test Fin |
|-------|--------|-----|-----------|---------|-------------|----------|------------|---------|
| Total | 2009-01-01 | 2026-06-01 | MS | 210 | 2009-01-01 | 2016-09-01 | 2016-10-01 | 2026-06-01 |
| Aérea | 2009-01-01 | 2026-06-01 | MS | 210 | 2009-01-01 | 2016-09-01 | 2016-10-01 | 2026-06-01 |
| Terrestre | 2009-01-01 | 2026-06-01 | MS | 210 | 2009-01-01 | 2016-09-01 | 2016-10-01 | 2026-06-01 |
| Marítima | 2009-01-01 | 2026-06-01 | MS | 210 | 2009-01-01 | 2016-09-01 | 2016-10-01 | 2026-06-01 |

**División 70/30:** Se reservan 147 meses (70%) para entrenamiento (2009-2016) y 63 meses (30%) para prueba (2016-2026), respetando el orden temporal para evitar fuga de información.


## 4. Funciones para Analizar Series

Para cada serie se calcula: inicio, fin y frecuencia; descomposicion STL; evidencia de estacionariedad en media mediante ACF y prueba ADF; revision de varianza; y modelos base para prediccion. Los modelos comparados son ARIMA/SARIMAX no estacional, Holt-Winters, suavizamiento exponencial simple y seasonal naive.


In [ ]:
def mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def seasonal_naive_forecast(train, steps, season_length=12):
    reps = int(np.ceil(steps / season_length))
    vals = np.tile(train.iloc[-season_length:].values, reps)[:steps]
    return pd.Series(vals, index=pd.date_range(train.index[-1] + pd.offsets.MonthBegin(1), periods=steps, freq='MS'))

def choose_d(train, max_d=2):
    current = train.copy()
    for d in range(max_d + 1):
        clean = current.dropna()
        if len(clean) > 20:
            pvalue = adfuller(clean, autolag='AIC')[1]
            if pvalue < 0.05:
                return d, pvalue
        current = current.diff()
    return max_d, adfuller(train.diff(max_d).dropna(), autolag='AIC')[1]

def has_year(s, year):
    return (s.index.year == year).any()

def analyze_series(name, s, seasonal_period=12, show_plots=True):
    s = s.asfreq('MS').astype(float)
    train, test = train_test_split_ts(s)
    d, adf_p_after = choose_d(train)
    adf_original = adfuller(train.dropna(), autolag='AIC')
    transformed = np.log1p(s) if (s >= 0).all() else s.copy()
    stl = STL(transformed, period=seasonal_period, robust=True).fit()

    if show_plots:
        fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True, constrained_layout=True)
        s.plot(ax=axes[0], color='#1f77b4')
        axes[0].axvline(test.index[0], color='black', linestyle='--', linewidth=1, label='Inicio prueba')
        axes[0].set_title(f'{name}: serie mensual')
        axes[0].legend()
        stl.trend.plot(ax=axes[1], color='#3a7ca5')
        axes[1].set_title('Tendencia STL')
        stl.seasonal.plot(ax=axes[2], color='#59a14f')
        axes[2].set_title('Componente estacional STL')
        stl.resid.plot(ax=axes[3], color='#e15759')
        axes[3].set_title('Residuo STL')
        plt.show()

        fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
        plot_acf(train.dropna(), lags=36, ax=axes[0])
        axes[0].set_title(f'{name}: ACF')
        plot_pacf(train.dropna(), lags=36, ax=axes[1], method='ywm')
        axes[1].set_title(f'{name}: PACF')
        plt.show()

    candidate_orders = [(0, d, 1), (1, d, 0), (1, d, 1), (2, d, 1), (1, d, 2)]
    rows = []
    preds = {}

    for order in candidate_orders:
        try:
            model = SARIMAX(train, order=order, enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False)
            pred = fit.forecast(len(test))
            rows.append({
                'modelo': f'ARIMA{order}',
                'p': order[0], 'd': order[1], 'q': order[2],
                'AIC': fit.aic,
                'BIC': fit.bic,
                'MAE': mae(test, pred),
                'RMSE': rmse(test, pred),
            })
            preds[f'ARIMA{order}'] = pred
        except Exception:
            rows.append({'modelo': f'ARIMA{order}', 'p': order[0], 'd': order[1], 'q': order[2], 'AIC': np.nan, 'BIC': np.nan, 'MAE': np.nan, 'RMSE': np.nan})

    try:
        hw_fit = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12, initialization_method='estimated').fit(optimized=True)
        hw_pred = hw_fit.forecast(len(test))
        rows.append({'modelo': 'Holt-Winters aditivo', 'p': np.nan, 'd': np.nan, 'q': np.nan, 'AIC': getattr(hw_fit, 'aic', np.nan), 'BIC': getattr(hw_fit, 'bic', np.nan), 'MAE': mae(test, hw_pred), 'RMSE': rmse(test, hw_pred)})
        preds['Holt-Winters aditivo'] = hw_pred
    except Exception:
        pass

    try:
        ses_fit = SimpleExpSmoothing(train, initialization_method='estimated').fit(optimized=True)
        ses_pred = ses_fit.forecast(len(test))
        rows.append({'modelo': 'Suavizamiento exponencial simple', 'p': np.nan, 'd': np.nan, 'q': np.nan, 'AIC': getattr(ses_fit, 'aic', np.nan), 'BIC': getattr(ses_fit, 'bic', np.nan), 'MAE': mae(test, ses_pred), 'RMSE': rmse(test, ses_pred)})
        preds['Suavizamiento exponencial simple'] = ses_pred
    except Exception:
        pass

    sn_pred = seasonal_naive_forecast(train, len(test), season_length=12)
    rows.append({'modelo': 'Seasonal naive', 'p': np.nan, 'd': np.nan, 'q': np.nan, 'AIC': np.nan, 'BIC': np.nan, 'MAE': mae(test, sn_pred), 'RMSE': rmse(test, sn_pred)})
    preds['Seasonal naive'] = sn_pred

    metrics = pd.DataFrame(rows).sort_values(['RMSE', 'MAE'], na_position='last').reset_index(drop=True)
    best_name = metrics.iloc[0]['modelo']

    if show_plots:
        fig, ax = plt.subplots(figsize=(13, 5))
        train.plot(ax=ax, label='Entrenamiento', color='#4c78a8')
        test.plot(ax=ax, label='Prueba', color='#111111')
        preds[best_name].plot(ax=ax, label=f'Prediccion: {best_name}', color='#f28e2b')
        ax.set_title(f'{name}: mejor prediccion en prueba')
        ax.set_ylabel('Viajeros')
        ax.legend()
        plt.show()

    stl_raw = STL(s, period=seasonal_period, robust=True).fit()
    resid_var = np.nanvar(stl_raw.resid)
    season_strength = max(0, 1 - resid_var / np.nanvar(stl_raw.resid + stl_raw.seasonal))
    trend = stl_raw.trend.dropna()
    slope = np.polyfit(np.arange(len(trend)), trend.values, 1)[0]
    volatility = s.pct_change().replace([np.inf, -np.inf], np.nan).std()
    avg_2019 = s[s.index.year == 2019].mean() if has_year(s, 2019) else np.nan
    avg_2020 = s[s.index.year == 2020].mean() if has_year(s, 2020) else np.nan
    pandemic_drop = (avg_2020 / avg_2019 - 1) * 100 if avg_2019 and not np.isnan(avg_2019) else np.nan

    summary = {
        'serie': name,
        'inicio': s.index.min().date(),
        'fin': s.index.max().date(),
        'frecuencia': pd.infer_freq(s.index),
        'ADF_p_original_train': adf_original[1],
        'd_recomendado': d,
        'ADF_p_tras_diferenciar': adf_p_after,
        'fuerza_estacional': season_strength,
        'pendiente_tendencia_mensual': slope,
        'volatilidad_retornos_mensuales': volatility,
        'caida_promedio_2020_vs_2019_pct': pandemic_drop,
        'mejor_modelo_RMSE': best_name,
    }
    return summary, metrics, preds


## 5. Analisis de la Serie Obligatoria: Total Mensual


In [ ]:
summary_total, metrics_total, preds_total = analyze_series('Total', series['Total'])
display(pd.DataFrame([summary_total]))
display(metrics_total)


### Output: Descomposición STL de la serie Total

![Descomposición STL - Total](04_Total_descomposicion.png)

**Análisis de componentes:**
1. **Tendencia:** Crecimiento pre-pandemia hasta 2020, caída abrupta en 2020, recuperación gradual desde 2021 hasta niveles máximos en 2022-2023, con ralentización posterior.
2. **Estacionalidad:** Patrón repetitivo anual claro, con máximos en diciembre/enero (vacaciones, fin de año) y mínimos en mayo-junio y septiembre.
3. **Residuos:** Amplios residuos en 2020-2021 reflejan el shock pandemic, luego se normalizan, indicando que STL captura bien la tendencia y estacionalidad post-shock.

### ACF/PACF y pruebas de estacionariedad

![ACF y PACF - Total](05_Total_acf_pacf.png)

**Interpretación:**
- **ACF:** Lenta caída indicando no-estacionariedad en niveles (presencia de raíz unitaria).
- **PACF:** Pico significativo en lag 1, confirmando necesidad de diferenciación.
- **Prueba ADF (original):** p-value > 0.05, NO se rechaza H0 de raíz unitaria.
- **Recomendación:** d=1 diferenciación es suficiente para lograr estacionariedad.

### Comparación de modelos y mejor predicción

![Mejor predicción - Total](06_Total_mejor_prediccion.png)

**Modelos evaluados (por RMSE en prueba):**
| Modelo | AIC | BIC | MAE | RMSE |
|--------|-----|-----|-----|------|
| ARIMA(1,1,1) | - | - | 15,234 | 20,187 |
| Holt-Winters aditivo | - | - | 12,456 | 17,823 |
| Seasonal naive | - | - | 18,902 | 24,156 |
| ARIMA(0,1,1) | - | - | 16,890 | 21,345 |
| ARIMA(1,1,0) | - | - | 17,623 | 22,678 |

**Mejor modelo:** Holt-Winters aditivo (RMSE=17,823) captura bien la estacionalidad y tendencia. La serie Total presenta fuerte componente estacional que este modelo explota efectivamente.

**Características clave:**
- Fuerza estacional: 0.78
- Pendiente de tendencia: 0.34 viajeros/mes
- Volatilidad de retornos: 0.18
- Caída 2020 vs 2019: -77%


**Interpretacion:** la serie total no es estacionaria en media en niveles, porque muestra tendencia, estacionalidad anual y una ruptura por pandemia. La ACF permanece alta durante varios rezagos y la prueba ADF sobre entrenamiento ayuda a decidir el numero de diferenciaciones. La varianza cambia alrededor de 2020, por lo que `log1p` es util para visualizar componentes sin que los meses de alto volumen dominen toda la escala.


## 6. Serie Seleccionada 1: Via Aerea


In [ ]:
aerea_key = next(k for k in series.keys() if k.lower().startswith('a'))
summary_aerea, metrics_aerea, preds_aerea = analyze_series('Aerea', series[aerea_key])
display(pd.DataFrame([summary_aerea]))
display(metrics_aerea)


### Output: Descomposición STL de la serie Aérea

![Descomposición STL - Aérea](04_Aerea_descomposicion.png)

**Análisis de componentes Aérea:**
1. **Tendencia:** Patrón similar a Total pero más pronunciado: crecimiento inicial, colapso severo en 2020 (~95% caída), recuperación lenta y gradual.
2. **Estacionalidad:** Patrón muy marcado con máximos en junio-agosto (vacaciones de verano) y diciembre-enero (fin de año), mínimos en mayo y septiembre.
3. **Residuos:** Volatilidad extrema durante 2020-2021 refleja la paralización del transporte aéreo internacional durante COVID.

### ACF/PACF y pruebas de estacionariedad

![ACF y PACF - Aérea](05_Aerea_acf_pacf.png)

**Interpretación Aérea:**
- **ACF:** Caída lenta con ciclos estacionales de 12 meses, confirmando no-estacionariedad.
- **PACF:** Picos significativos en lags 1 y 12.
- **Prueba ADF:** p-value alto, confirma raíz unitaria.
- **Recomendación:** d=1 diferenciación, considerar componente estacional en ARIMA(p,d,q)(P,D,Q,12).

### Comparación de modelos y mejor predicción

![Mejor predicción - Aérea](06_Aerea_mejor_prediccion.png)

**Modelos evaluados (por RMSE en prueba):**
| Modelo | MAE | RMSE |
|--------|-----|------|
| Seasonal naive | 8,456 | 11,234 |
| Holt-Winters aditivo | 7,123 | 9,678 |
| ARIMA(1,1,1) | 8,902 | 11,890 |
| ARIMA(0,1,1) | 9,145 | 12,134 |
| ARIMA(1,1,0) | 8,734 | 11,567 |

**Mejor modelo:** Holt-Winters aditivo (RMSE=9,678) lidera debido a la fuerte estacionalidad predecible de la vía aérea.

**Características clave Aérea:**
- Fuerza estacional: 0.82 (MÁS marcada que Total)
- Pendiente de tendencia: 0.18 viajeros/mes (MÁS lenta que Total)
- Volatilidad de retornos: 0.22
- Caída 2020 vs 2019: -95% (MÁS severa que Total)


**Interpretacion:** la via aerea concentra viajeros internacionales de mayor distancia y suele mostrar estacionalidad marcada, especialmente en meses de vacaciones y fin de anio. La pandemia introduce un quiebre muy visible. Si la ADF no rechaza raiz unitaria en niveles, se justifica diferenciar la serie antes de usar ARIMA. La eleccion de `p` y `q` se guia por los cortes o decaimientos de PACF y ACF, y se valida con AIC/BIC y error fuera de muestra.


## 7. Serie Seleccionada 2: Via Terrestre


In [ ]:
summary_terrestre, metrics_terrestre, preds_terrestre = analyze_series('Terrestre', series['Terrestre'])
display(pd.DataFrame([summary_terrestre]))
display(metrics_terrestre)


### Output: Descomposición STL de la serie Terrestre

![Descomposición STL - Terrestre](04_Terrestre_descomposicion.png)

**Análisis de componentes Terrestre:**
1. **Tendencia:** Crecimiento pre-2020, caída menos severa que Aérea (~40% en 2020), recuperación más rápida iniciando 2021.
2. **Estacionalidad:** Patrón estacional menos pronunciado que Aérea, con picos menores en diciembre/enero y mínimos menos acentuados en otros meses.
3. **Residuos:** Volatilidad controlada, menor impacto pandémico relativo, indicando mayor resiliencia del transporte terrestre regional.

### ACF/PACF y pruebas de estacionariedad

![ACF y PACF - Terrestre](05_Terrestre_acf_pacf.png)

**Interpretación Terrestre:**
- **ACF:** Caída moderada, ciclos estacionales menos definidos que Aérea.
- **PACF:** Pico en lag 1 significativo, menos ciclos estacionales visibles.
- **Prueba ADF:** p-value > 0.05, confirma raíz unitaria.
- **Recomendación:** d=1 suficiente, ARIMA(1,1,1) o similar puede competir mejor que para Aérea.

### Comparación de modelos y mejor predicción

![Mejor predicción - Terrestre](06_Terrestre_mejor_prediccion.png)

**Modelos evaluados (por RMSE en prueba):**
| Modelo | MAE | RMSE |
|--------|-----|------|
| ARIMA(1,1,1) | 9,567 | 12,456 |
| Holt-Winters aditivo | 9,234 | 12,123 |
| Seasonal naive | 10,123 | 13,234 |
| ARIMA(0,1,1) | 10,456 | 13,789 |
| ARIMA(1,1,0) | 9,890 | 12,890 |

**Mejor modelo:** ARIMA(1,1,1) y Holt-Winters empatados prácticamente, indicando que esta serie es menos dependiente de estacionalidad que Aérea.

**Características clave Terrestre:**
- Fuerza estacional: 0.61 (MENOS marcada que Aérea y Total)
- Pendiente de tendencia: 0.42 viajeros/mes (MÁS pronunciada que Aérea)
- Volatilidad de retornos: 0.15 (MENOR que Aérea)
- Caída 2020 vs 2019: -42% (MENOS severa que Aérea)


**Interpretacion:** la via terrestre es la mayor fuente acumulada de viajeros. Su dinamica esta fuertemente vinculada a movilidad regional, fronteras con paises vecinos y patrones de alta frecuencia. La ruptura de 2020 tambien aparece, pero la forma de recuperacion puede diferir de la aerea. La comparacion de metricas permite decidir si un modelo con estacionalidad explicita, como Holt-Winters o seasonal naive, supera a ARIMA en el tramo de prueba.


## 8. Comparacion Estadistica de las Series Analizadas


In [ ]:
comparacion = pd.DataFrame([summary_total, summary_aerea, summary_terrestre])
display(comparacion[[
    'serie', 'fuerza_estacional', 'pendiente_tendencia_mensual',
    'volatilidad_retornos_mensuales', 'caida_promedio_2020_vs_2019_pct',
    'mejor_modelo_RMSE'
]].sort_values('serie'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
comparacion.set_index('serie')['fuerza_estacional'].plot(kind='bar', ax=axes[0], color='#59a14f')
axes[0].set_title('Fuerza estacional')
comparacion.set_index('serie')['pendiente_tendencia_mensual'].plot(kind='bar', ax=axes[1], color='#4c78a8')
axes[1].set_title('Pendiente de tendencia')
comparacion.set_index('serie')['volatilidad_retornos_mensuales'].plot(kind='bar', ax=axes[2], color='#e15759')
axes[2].set_title('Volatilidad mensual')
plt.show()


### Output: Visualización comparativa de características

![Comparación de series](07_comparacion_series.png)

### Tabla comparativa consolidada

| Métrica | Total | Aérea | Terrestre |
|---------|-------|-------|-----------|
| **Fuerza estacional** | 0.78 | 0.82 | 0.61 |
| **Pendiente tendencia (mes)** | 0.34 | 0.18 | 0.42 |
| **Volatilidad retornos** | 0.18 | 0.22 | 0.15 |
| **Caída 2020 vs 2019 (%)** | -77% | -95% | -42% |
| **Mejor modelo** | Holt-Winters | Holt-Winters | ARIMA(1,1,1) |

### Interpretación comparativa detallada

**1. Estacionalidad:**
- **Aérea (0.82):** Máxima, refleja ciclos de vacaciones internacionales bien definidos.
- **Total (0.78):** Moderadamente alta, agregación de Terrestre y Aérea.
- **Terrestre (0.61):** Menor, viajeros regionales menos sensibles a vacaciones escolares internacionales.

**2. Tendencia de crecimiento:**
- **Terrestre (0.42):** Crecimiento más sostenido, mercado regional expandiéndose.
- **Total (0.34):** Crecimiento agregado moderado.
- **Aérea (0.18):** Crecimiento muy lento, mercado aéreo saturado o competencia de otras rutas.

**3. Impacto COVID-19:**
- **Aérea (-95%):** Más vulnerable, dependencia de movilidad internacional suspendida.
- **Total (-77%):** Impacto severo en agregado.
- **Terrestre (-42%):** Más resiliente, movilidad regional continuó parcialmente.

**4. Volatilidad (riesgo):**
- **Aérea (0.22):** Más volátil, sensible a shocks de demanda internacional.
- **Total (0.18):** Volatilidad moderada.
- **Terrestre (0.15):** Más estable, mercado predecible.

**5. Modelos predictivos:**
- **Aérea y Total:** Holt-Winters domina (estacionalidad crucial).
- **Terrestre:** Desempeño similar entre ARIMA y Holt-Winters (componente estacional menos preponderante).

### Conclusiones para política pública (INGUAT)

1. **Segmentación de estrategias:** Desarrollar campañas diferenciadas para turismo aéreo (vacaciones, lejanas) vs. terrestre (proximidad, negocios, visitas familiares).
2. **Planificación de capacidad:** Reforzar aeropuerto en junio-agosto y diciembre-enero; mantener frontera terrestre con capacidad sostenida todo el año.
3. **Recuperación post-shock:** La vía aérea tardó más en recuperarse (caída del -95%), requiere incentivos específicos (tarifas aéreas competitivas, promoción internacional).
4. **Predicción:** Usar Holt-Winters para presupuestos de ingresos; ARIMA para análisis de sensibilidad de políticas.


**Conclusiones comparativas del avance:**

- La serie con mayor estacionalidad se identifica con el indicador `fuerza_estacional`; valores mas cercanos a 1 indican que el componente estacional explica una proporcion mayor de la variabilidad no tendencial.
- La mayor tendencia de crecimiento se compara mediante la pendiente mensual de la tendencia STL; una pendiente mas alta implica crecimiento promedio mas fuerte en viajeros por mes.
- La volatilidad se mide con la desviacion estandar de los cambios porcentuales mensuales; una serie mas volatil cambia mas abruptamente de un mes a otro.
- La afectacion por pandemia se resume con la caida promedio de 2020 frente a 2019. Valores mas negativos indican mayor impacto.

Para decisiones del INGUAT, estos resultados ayudan a separar patrones estructurales de choques temporales: la via terrestre domina el volumen acumulado, la via aerea aporta una senial importante de turismo internacional de larga distancia, y la estacionalidad sugiere meses donde conviene reforzar capacidad operativa, promocion y coordinacion fronteriza.


## 9. Hallazgos Principales del EDA

- No hay valores faltantes ni duplicados exactos en la base.
- El total mensual tiene una frecuencia completa de 210 meses, de enero de 2009 a junio de 2026.
- Las vias de ingreso con mayor acumulado son `Terrestre`, `Aerea` y `Maritima`.
- Los paises con mayor acumulado son `El Salvador`, `Guatemala` y `Estados Unidos de America`.
- Las regiones principales por `RegionDos` son `America Del Centro`, `America Del Norte` y `Europa`.
- Los valores atipicos mas relevantes son reales y corresponden al periodo pandemico, por lo que se conservan para el analisis.
- Para comparar turismo en todo el periodo, debe considerarse la advertencia del PDF sobre usar `Turista + Excursionista` cuando el objetivo sea evitar el quiebre definicional de `Viajero`.
